# Create grid points for kriging

**Yuxi Zhang** · Oct 18, 2026 · [@yuxi-research](https://github.com/yuxi-research)


📘 This notebook creates an empty gridded raster file filled with `np.nan` values within the buffered farm boundary. The raster is used as the target grid for aligning rasters from other data sources and for kriging.

⚙️ **Main Processing Steps**

1. **Process**

   * Read the farm boundary shapefile.
   * Create a buffer around the farm boundary.
   * Create gridded points within the total bounds of the buffered boundary.
   * Convert the GeoDataFrame to an 2D array containing all np.nan values.
   * Create a raster from the 2D array and save it as a GeoTIFF

2. **Test**
   * Test the exported GeoTIFF file.

💬 **Notes**

📥 **Input**

* `./data_lv1/shapefiles/`

  * `{farm_name}_boundary_epsg{proj_crs}.shp`: Farm boundary in a projected CRS.
* `settings.py`: Configuration file specifying the farm name, target CRS, folder structure, and file paths.

📤 **Output**

* `./data_lv2/shapefiles/`

  * `target_grid.tif`: Empty gridded raster file.
  * `target_grid_test.tif`: Empty gridded raster file used for testing.

🧩 **Parameters**

* `grid_spacing`: Specifies the output raster cell size.
* `buffer`: Specifies the buffer distance between the output raster boundary and the farm boundary.
* `mode`: If `mode` is set to `'test'`, a coarse-resolution raster is created for testing the B2PK notebook.


### Settings

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from affine import Affine
import rasterio
import rioxarray as rxr


In [ ]:
# Path settings
from global_settings import ProjectPaths, GRID_SPACING, BUFFER_DIS
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc

#### Pipeline parameters

In [ ]:
# Parameters
try:
    grid_spacing
except NameError:
    grid_spacing = GRID_SPACING

try:
    buffer
except NameError:
    buffer = BUFFER_DIS

try:
    mode
except NameError:
    mode = ''  # test mode to generate lower resolution outputs

### Read data

In [ ]:
# Read the boundary shapefile
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))

# Create buffer around the boundary
bound_buffer = bound.copy()
bound_buffer.geometry = bound_buffer.geometry.buffer(buffer)  # buffer m buffer

### Create gridded points within the total bounds of the buffered boundary. 

In [ ]:
# Create grid points within total bounds
xmin, ymin, xmax, ymax = bound_buffer.total_bounds

# Round to the nearest grid_spacing to avoid unnecessary floating point number issues
xmin, ymin, xmax, ymax = (np.floor(xmin / grid_spacing) * grid_spacing, np.floor(ymin / grid_spacing) * grid_spacing, 
                            np.ceil(xmax / grid_spacing) * grid_spacing, np.ceil(ymax / grid_spacing) * grid_spacing)

# Create grid points
x_coords = np.arange(xmin, xmax, grid_spacing)
y_coords = np.arange(ymin, ymax, grid_spacing)
grid_points = [Point(x, y) for x in x_coords for y in y_coords]
grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs=bound_buffer.crs)
# grid_gdf.to_file(os.path.join(pth_out1_shp, f'{farm_name}_gridpoints_epsg{proj_crs}.shp'))

# Filter grid points within the buffered boundary
grid_within_bound = gpd.sjoin(grid_gdf, bound_buffer, how='inner', predicate='within')
grid_within_bound = grid_within_bound.drop(columns=['index_right'])
grid_within_bound['x'] = grid_within_bound.geometry.x
grid_within_bound['y'] = grid_within_bound.geometry.y
grid_within_bound = grid_within_bound.reset_index(drop=True)
grid_within_bound['area'] = grid_spacing * grid_spacing

# Export the grid points within the boundary and the full extent grid points. Buffer distance applied.
# grid_within_bound.to_file(os.path.join(pth_out1_shp, f'{farm_name}_gridpoints_epsg{proj_crs}.shp'))

In [ ]:
# Visualize the grid points within the boundary
fig, ax = plt.subplots(figsize=(8, 8))
bound.boundary.plot(ax=ax, color='black')
# bound_buffer.boundary.plot(ax=ax, color='grey', linestyle='--')

grid_gdf.plot(ax=ax, color='blue', alpha=0.3, markersize=1, label='Grid full extent')
grid_within_bound.plot(ax=ax, color='red', alpha=0.3, markersize=1, label='Grid within boundary')
plt.title('Grid Points within Boundary')
plt.xlabel('Easting (m)')
plt.ylabel('Northing (m)')
plt.legend(frameon=True, facecolor='white', framealpha=0.5, loc=inset_loc)
plt.show()

### Convert the GeoDataFrame to an 2D array containing all np.nan values.


In [ ]:
# Convert the full extent grid points to a GeoDataFrame for further processing
raster_to_export = grid_gdf

In [ ]:
# Create a 2D array containing all np.nan values with the same shape as the grid.
nrows = raster_to_export.geometry.y.nunique()
ncols = raster_to_export.geometry.x.nunique()
xmin = raster_to_export.geometry.x.min()
xmax = raster_to_export.geometry.x.max()
ymin = raster_to_export.geometry.y.min()
ymax = raster_to_export.geometry.y.max()

x_res = (xmax - xmin) / (ncols - 1)
y_res = (ymax - ymin) / (nrows - 1)
print(nrows, ncols, x_res, y_res)

raster_to_export['x'] = raster_to_export.geometry.x
raster_to_export['y'] = raster_to_export.geometry.y

# Create a meshgrid
xx, yy = np.meshgrid(np.linspace(xmin, xmax, ncols), np.linspace(ymin, ymax, nrows))

# Fill the grid with NaNs
z_2d = np.full((nrows, ncols), np.nan)
z_2d.shape

# for kriged points, find the corresponding row and column in the grid and fill the values
for x, y in raster_to_export[['x', 'y']].values:
    col = int((x - xmin) / x_res)
    row = int((ymax - y) / y_res)
    if 0 <= row < nrows and 0 <= col < ncols:
        z_2d[row, col] = 0

# Define the transform
transform = Affine.translation(xmin - x_res / 2,
                                ymax + y_res / 2) * Affine.scale(x_res, -y_res)

In [ ]:
# Create a raster from the 2D array and save it as a GeoTIFF
if 'test' in mode.lower():
    outfilename = 'target_grid_test.tif'
else:
    outfilename = 'target_grid.tif'
with rasterio.open(dirs.lv2_data / outfilename,
    'w',
    driver='GTiff',
    height=nrows,
    width=ncols,
    count=1,
    dtype=z_2d.dtype,
    crs=f'EPSG:{proj_crs}',
    transform=transform,
    nodata=np.nan
) as dst:
    dst.write(z_2d, 1)


### Check

In [ ]:
import utils.check_data as check
check.check_raster(str(dirs.lv2_data / outfilename))